# Sola Face LoRA (SDXL) — Colab

1. **Runtime → GPU** (T4 ок, L4/A100 краще)
2. Завантаж zip з `datasets/sola_face_kohya` (локально зроби zip папки)
3. Run all
4. Скачай `sola_face_sdxl.safetensors` з outputs

Trigger: `sola_face`


In [ ]:
# @title 1) Setup Kohya sd-scripts
import os, sys, subprocess
from google.colab import files

os.chdir("/content")
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121 || true
if not os.path.isdir("/content/sd-scripts"):
    !git clone --depth 1 https://github.com/kohya-ss/sd-scripts /content/sd-scripts
os.chdir("/content/sd-scripts")
!pip -q install -r requirements.txt
!pip -q install bitsandbytes prodigyopt lion-pytorch
print("OK setup")


In [ ]:
# @title 2) Upload sola_face_kohya.zip
import os, zipfile, shutil
from google.colab import files

DATA = "/content/sola_data"
os.makedirs(DATA, exist_ok=True)
os.chdir(DATA)
print("Upload sola_face_kohya.zip (must contain 10_sola_face/ with jpg+txt)")
uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(DATA)
print("Tree:")
for root, dirs, fs in os.walk(DATA):
    level = root.replace(DATA, "").count(os.sep)
    if level > 3: continue
    print("  " * level + os.path.basename(root) + "/")
    if level >= 2: continue
    for f in fs[:8]:
        print("  " * (level + 1) + f)


In [ ]:
# @title 3) Find train_data_dir (folder that CONTAINS 10_sola_face)
import os

def find_train_root(base):
    for root, dirs, files in os.walk(base):
        if "10_sola_face" in dirs:
            return root
    raise FileNotFoundError("10_sola_face not found — zip structure wrong")

TRAIN_ROOT = find_train_root("/content/sola_data")
OUT = "/content/outputs/sola_face_lora"
os.makedirs(OUT, exist_ok=True)
print("TRAIN_ROOT =", TRAIN_ROOT)
print("images =", len([f for f in os.listdir(os.path.join(TRAIN_ROOT, "10_sola_face")) if f.lower().endswith((".jpg",".jpeg",".png"))]))


In [ ]:
# @title 4) Train SDXL LoRA (face)
import os
os.chdir("/content/sd-scripts")

# Base: SDXL 1.0 from HF (reliable). Later you can swap to CyberRealistic path.
cmd = f'''
accelerate launch --num_cpu_threads_per_process 2 sdxl_train_network.py \\
  --pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 \\
  --train_data_dir={TRAIN_ROOT} \\
  --output_dir={OUT} \\
  --output_name=sola_face_sdxl \\
  --save_model_as=safetensors \\
  --caption_extension=.txt \\
  --resolution=1024,1024 \\
  --enable_bucket \\
  --min_bucket_reso=512 \\
  --max_bucket_reso=2048 \\
  --train_batch_size=1 \\
  --max_train_epochs=10 \\
  --save_every_n_epochs=2 \\
  --learning_rate=1e-4 \\
  --unet_lr=1e-4 \\
  --text_encoder_lr=5e-5 \\
  --lr_scheduler=cosine_with_restarts \\
  --lr_scheduler_num_cycles=3 \\
  --optimizer_type=AdamW8bit \\
  --network_module=networks.lora \\
  --network_dim=16 \\
  --network_alpha=8 \\
  --mixed_precision=fp16 \\
  --xformers \\
  --cache_latents \\
  --cache_latents_to_disk \\
  --seed=42 \\
  --keep_tokens=1 \\
  --noise_offset=0.0357 \\
  --min_snr_gamma=5 \\
  --max_data_loader_n_workers=2
'''
print(cmd)
!{cmd}


In [ ]:
# @title 5) Download LoRA
import os, glob
from google.colab import files

paths = sorted(glob.glob("/content/outputs/sola_face_lora/*.safetensors"))
print("Found:", paths)
assert paths, "No LoRA produced — check train logs"
# prefer final name
final = [p for p in paths if p.endswith("sola_face_sdxl.safetensors")] or paths[-1:]
for p in final:
    print("Downloading", p)
    files.download(p)


### У Sunside / Fooocus
Поклади файл у `models/loras/`, Character **Sola**, у prompt: `sola_face, ...` weight ~0.8–1.0
